<div style="background-color:#d4edda;font-size:19px; color:#155724; padding:18px; border-radius:10px; border:1px solid #c3e6cb; font-family:Arial, sans-serif;">

<center> <b>Réalisé par A-ABO
<br>
<center> Ce projet a été réalisé par un étudiant en 2026.

<br>

<div style="font-size:9px; color:#4a5c4f; font-style:italic; text-align:center;">

<center> Ce projet peut contenir des erreurs.

</div>

</div>

<center> <h2> TD et TME 9: CREATION DE SCHEMAS-MODIFICATION DES DONNEES


# Installation H2 

Le système utilisé pendant les TME est H2.

In [14]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


Télécharger le pilote de H2

**Relancez le kernel**: Kernel -> Restart kernel ...

https://www.psycopg.org/docs/
http://localhost:8082

In [15]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

**Attention**: vous pouvez cliquer sur les 3 lignes dans la fenêtre de gauche pour d'afficher les différentes sections du notebook   

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [16]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [17]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Tester si le serveur a été démarré

In [19]:
%%bash
nc -zv 127.0.0.1 8082

bash: line 1: nc: command not found


CalledProcessError: Command 'b'nc -zv 127.0.0.1 8082\n'' returned non-zero exit status 127.

# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [20]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [21]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.

In [22]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor
print('fonction définie')

fonction définie


# Affichage du schéma d'une table

In [23]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)
    

# Affichage du schéma global 

In [24]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)
    

# Exercices TD

pour supprimer le schéma de la base courante

In [25]:
query="""
DROP ALL OBJECTS
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611525c0; closed: -1>

## VILLES ET PAYS
On veut créer un schéma relationnel pour stocker des informations sur des villes et des pays.



### Traduisez le schéma relationnel suivant en instructions SQL:


- Ville(<u>nom</u>, population, pays*) 
- Pays(<u>nom</u>, capitale*)

où pays est une référence vers un pays dans la table Pays et capitale est une référence vers une ville dans la table Ville.

In [26]:
query="""
create table Ville(
    nom varchar(20) primary key,
    pipulation numeric(10),
    pays varchar(20)
);
create table Pays(
    nom varchar(20) primary key,
    capital varchar(20) 

)

"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461137d30; closed: -1>

In [27]:
query="""
alter table Ville add constraint fk_ville_pays foreign key (pays) references Pays(nom);
alter table Pays add constraint fk_pays_ville foreign key (capital) references Ville(nom);
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461134c70; closed: -1>

Interroger le catalogue de données

In [28]:
show_schema(connection)

*********** Tables ************
2 rows, display only 1000 rows
|table_name|
|-----------
|ville     |
|pays      |


*********** Domaines ************
0 rows


*********** Attributs ************
5 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|ville     |nom        |character varying|None          |NO         |20                      |None             |
|ville     |pipulation |numeric          |None          |YES        |None                    |10               |
|ville     |pays       |character varying|None          |YES        |20                      |None             |
|pays      |nom        |character varying|None          |NO         |20                      |None             |
|pays      |capital    |character varying|None          |YES        |20                      |No

### Insérez la France avec sa capitale Paris (3 millions d'habitants) dans la base de données (deux insertions).

In [29]:
query="""
insert into Pays values('France' ,null);
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461180b80; closed: -1>

In [30]:
query="""
insert into ville values('Paris' , 300000, 'France');
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611809a0; closed: -1>

In [31]:
query="""
update pays 
set capital = 'Paris'
where nom = 'France'
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [32]:
query="""
select * from Ville
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom  |pipulation|pays  |
|------------------------
|Paris|300000    |France|


<cursor object at 0x7ff461180130; closed: -1>

In [33]:
query="""
select * from Pays
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|nom   |capital|
|---------------
|France|Paris  |


<cursor object at 0x7ff461180f40; closed: -1>

### Modifiez le schéma de telle manière que la suppression d'un pays déclenche automatiquement la suppression de toutes les villes du pays.

In [34]:
query="""
alter table ville 
drop constraint fk_ville_pays
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461180a90; closed: -1>

In [35]:
query="""
alter table ville add constraint fk_ville_pays 
foreign key (pays) references Pays(nom) on delete cascade;
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181120; closed: -1>

###  Effacez les Pays 

In [36]:
query="""
delete from pays
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [37]:
query="""
select * from Pays
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611808b0; closed: -1>

### Exécutez la requête suivante; pourquoi Paris a disparu ?

-->>> on delete CASCADE !!!

In [38]:
query="""
select * from Ville
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181300; closed: -1>

In [39]:
show_schema(connection)

*********** Tables ************
2 rows, display only 1000 rows
|table_name|
|-----------
|ville     |
|pays      |


*********** Domaines ************
0 rows


*********** Attributs ************
5 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|ville     |nom        |character varying|None          |NO         |20                      |None             |
|ville     |pipulation |numeric          |None          |YES        |None                    |10               |
|ville     |pays       |character varying|None          |YES        |20                      |None             |
|pays      |nom        |character varying|None          |NO         |20                      |None             |
|pays      |capital    |character varying|None          |YES        |20                      |No

## MISE À JOUR DE TABLES

Considérer le schéma Entreprise du TD précédent rappelé ici :

- EMPLOYE (<u>NumSS</u>, NomE, PrenomE, NumChef*, VilleE, DateNaiss) 
- PROJET(<u>NumProj</u>, NomProj, RespProj*, VilleP, Budget) 
- EMBAUCHE (<u>NumSS*, NumProj*</u>, DateEmb, Profil*) 
- GRILLE_SAL (<u>Profil</u>, salaire)

La clé primaire de chaque relation est soulignée et les attributs des clés étrangères sont suivis d’un astérisque. Cette base contient des informations sur des employés et sur les projets dans lesquels ils sont impliqués. Ces employés sont embauchés dans un projet sur un profil donné et perçoivent un salaire en fonction de ce profil. Le chef d'un employé dans la table Employé et le chef d'un projet dans la table Projet sont des employés.

Le but de cet exercice est d'exprimer des instructions permettant d'insérer, de modifier et de supprimer des nuplets.

Dire à chaque fois si l'instruction exprimée est acceptée ou rejetée par le système en justifiant.

In [40]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
connection.commit()

0 rows
0 rows


In [41]:
show_schema(connection)

*********** Tables ************
4 rows, display only 1000 rows
|table_name|
|-----------
|embauche  |
|employe   |
|projet    |
|grille_sal|


*********** Domaines ************
5 rows, display only 1000 rows
|domain_name|check_clause                                 |
|----------------------------------------------------------
|dsal       |VALUE <= 90000                               |
|ddatenaiss |DATEDIFF(YEAR, VALUE, CURRENT_DATE) <= 70    |
|dvilles    |LOWER(VALUE) IN('paris', 'lyon', 'marseille')|
|dnumproj   |CHAR_LENGTH(VALUE) >= 5                      |
|dnumss     |CHAR_LENGTH(VALUE) = 5                       |


*********** Attributs ************
17 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|embauche  |numss      |numeric          |None          |NO         |None    

### INSERTIONS

In [42]:
query="""
SHOW COLUMNS FROM Employe;
SHOW COLUMNS FROM PROJET;
SHOW COLUMNS FROM GRILLE_SAL;
SHOW COLUMNS FROM EMBAUCHE
"""
execute(connection,query,show=True)

4 rows, display only 10 rows
|field  |type                 |null|key|default|
|-----------------------------------------------
|numss  |NUMERIC(5)           |NO  |PRI|NULL   |
|numproj|NUMERIC(7)           |NO  |PRI|NULL   |
|dateemb|DATE                 |YES |   |NULL   |
|profil |CHARACTER VARYING(20)|NO  |   |NULL   |


<cursor object at 0x7ff4611816c0; closed: -1>

##### Insérer l'employé identifié par '12456' qui se prénomme 'Alain'.


In [43]:
query="""
insert into employe values ('12456','Alain',null,null,null,null)
"""
# select * from employe   <<--- to check 
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


##### Insérer l'employée identifiée par '21456' qui s'appelle 'LARS Anna', qui habite 'Paris' et qui est née le 25-08-1975.

In [44]:
query="""
insert into employe values ('21456', 'LARS' , 'Anna',null , 'Paris' , Date '1975-08-25')
"""
# select * from employe  <<--- to check 
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


##### Insérer le projet numéro '78143' dénommé 'ORCA' qui s'opère sous la responsabilité de 'Lars Anna' à Paris et qui a pour budget 250 000 euros.

In [45]:
query="""
insert into projet values ('78143' , 'ORCA' , '21456' , 'Paris' , 250000)
"""
# select * from projet    <<--- to check 
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


##### Renseigner dans la base les salaires correspondants aux profils suivants : ('Responsable',80000), ('Développeur', 45000), ('Technicien',35000)

In [46]:
query="""
insert into grille_sal values ('Responsable',80000)
"""
execute(connection,query,show=True)


1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [47]:
query="""
insert into grille_sal values ('Developpeur',45000)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [48]:
query="""
insert into grille_sal values ('Technicien',35000)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [49]:
query="""
select * from grille_sal
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|profil     |salaire |
|---------------------
|Responsable|80000.00|
|Developpeur|45000.00|
|Technicien |35000.00|


<cursor object at 0x7ff4611813f0; closed: -1>

##### Renseigner dans la base le fait que 'Alain' a été embauché dans le projet 'ORCA' en tant que testeur en date du 01-04-2014.

In [50]:
query="""
insert into embauche values ('12456' , '78143' , date'2014-04-01', 'testeur')
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('testeur')"; SQL statement:

insert into embauche values ('12456' , '78143' , date'2014-04-01', 'testeur' [23506-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('testeur')"; SQL statement:

insert into embauche values ('12456' , '78143' , date'2014-04-01', 'testeur' [23506-214]
' occurred


##### Renseigner dans la base le fait que 'LARS Anna' fut embauchée dans le projet 'MEDUSA' en date du 28-02-2012 en tant que 'Développeur'.

<div style=" color:green; font-style:italic; "> Cette insertion est impossible car le projet MEDUSA n’existe pas dans la table Projet.


### SUPPRESSIONS

##### Supprimer les employés de plus de 67 ans

In [51]:
query="""

delete from employe 
where datediff ('year',datenaiss,current_date) > 67

"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611817b0; closed: -1>

##### Supprimer les employés prénommés 'Alain'

In [52]:
query="""
delete from employe 
where prenome = 'Alain'
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611818a0; closed: -1>

##### Supprimer l'employée 'LARS Anna'

<div style=" color:green; font-style:italic; "> Cette suppression est impossible car l’employée est référencée comme responsable dans la table Projet (contrainte d’intégrité référentielle).


In [53]:
query="""
delete from employe 
where nome = 'LARS' and prenome = 'Anna'
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_resp: public.projet FOREIGN KEY(respproj) REFERENCES public.employe(numss) (CAST(21456 AS NUMERIC(5)))"; SQL statement:

delete from employe 
where nome = 'LARS' and prenome = 'Anna [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_resp: public.projet FOREIGN KEY(respproj) REFERENCES public.employe(numss) (CAST(21456 AS NUMERIC(5)))"; SQL statement:

delete from employe 
where nome = 'LARS' and prenome = 'Anna [23503-214]
' occurred


##### Supprimer les employés embauchés lors de la dernière année.

In [54]:
query="""
delete from embauche
where datediff(year,dateemb, current_date) <= 1
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181a80; closed: -1>

##### Supprimer les projets dont la proportion des salaires dépasse la moitié du budget.

In [55]:
query="""
DELETE FROM Projet
WHERE NumProj IN (
    SELECT e.NumProj
    FROM Embauche e
    JOIN Grille_Sal g ON e.Profil = g.Profil
    JOIN Projet p ON p.NumProj = e.NumProj
    GROUP BY e.NumProj, p.Budget
    HAVING SUM(g.Salaire) > p.Budget / 2
);
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181b70; closed: -1>

### MISES À JOUR

##### Désormais on connait que l'employé Alain porte le nom BERNARD. Répercuter cette information.

In [56]:
query="""
update employe 
set nome = 'BERNARD'
where nome = 'Alain'
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


##### L'employée LARS Anna est promue à la tête du projet identifié par 99245. Elle doit d'abord déménager avant de prendre ses responsabilités. Donner l'instruction qui modifie la ville de cette employée. On suppose qu'il existe un nuplet dans la table Projet avec les données suivantes : (99245,'MEDUSA',null,'Lyon',350000)

In [57]:
query="""
update employe
set villee = 'Lyon'
where nome = 'LARS' and prenome = 'Anne'
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181c60; closed: -1>

##### Le budget de chaque projet équivaut au double des salaires de ses employés. Donner une instruction qui modifie les budgets des projets en conséquence.

In [58]:
query="""
update projet p 
set budget =  2 * (
                select sum(g.salaire)
                from embauche e join grille_sal g on g.profil = e.profil
                where e.numproj = p.numproj
            )
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


# Exercice TME: partie 1


In [59]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

0 rows
0 rows
0 rows


In [60]:
show_schema(connection)

*********** Tables ************
4 rows, display only 1000 rows
|table_name|
|-----------
|embauche  |
|employe   |
|projet    |
|grille_sal|


*********** Domaines ************
5 rows, display only 1000 rows
|domain_name|check_clause                                 |
|----------------------------------------------------------
|dsal       |VALUE <= 90000                               |
|ddatenaiss |DATEDIFF(YEAR, VALUE, CURRENT_DATE) <= 70    |
|dvilles    |LOWER(VALUE) IN('paris', 'lyon', 'marseille')|
|dnumproj   |CHAR_LENGTH(VALUE) >= 5                      |
|dnumss     |CHAR_LENGTH(VALUE) = 5                       |


*********** Attributs ************
17 rows, display only 1000 rows
|table_name|column_name|data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|---------------------------------------------------------------------------------------------------------------
|embauche  |numss      |numeric          |None          |NO         |None    

## Insertions rejetées (voir TME8)

### Proposer une insertion dans la table Employé qui ne respecte pas la contrainte de clé primaire.

In [61]:
query="""
insert into employe values (22334,'t1','t2','22222',null,null)

"""
execute(connection,query,show=True)

The error 'Unique index or primary key violation: "public.PRIMARY_KEY_9 ON public.employe(numss) VALUES ( /* 1 */ CAST(22334 AS NUMERIC(5)) )"; SQL statement:

insert into employe values (22334,'t1','t2','22222',null,null [23505-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Unique index or primary key violation: "public.PRIMARY_KEY_9 ON public.employe(numss) VALUES ( /* 1 */ CAST(22334 AS NUMERIC(5)) )"; SQL statement:

insert into employe values (22334,'t1','t2','22222',null,null [23505-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de limite d'âge.

In [62]:
query="""
insert into employe values('22233',null,null,null,null,'1111-11-11')
"""
execute(connection,query,show=True)

The error 'Check constraint violation: "(DATEDIFF(YEAR, VALUE, CURRENT_DATE) <= 70)"; SQL statement:

insert into employe values('22233',null,null,null,null,'1111-11-11' [23513-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Check constraint violation: "(DATEDIFF(YEAR, VALUE, CURRENT_DATE) <= 70)"; SQL statement:

insert into employe values('22233',null,null,null,null,'1111-11-11' [23513-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de longueur de l'attribut NumSS.

In [63]:
query="""
insert into employe values ('22222222222' , null , null ,null,null,null)
"""
execute(connection,query,show=True)

The error 'Value too long for column "numss NUMERIC(5)": "'22222222222' (11)"; SQL statement:

insert into employe values ('22222222222' , null , null ,null,null,null [22001-214]
DETAIL:  org.h2.jdbc.JdbcSQLDataException: Value too long for column "numss NUMERIC(5)": "'22222222222' (11)"; SQL statement:

insert into employe values ('22222222222' , null , null ,null,null,null [22001-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte sur les villes possibles.

In [64]:
query="""
insert into employe values('98765', null, null,null,'DeadLand',null)
"""
execute(connection,query,show=True)

The error 'Check constraint violation: "(LOWER(VALUE) IN('paris', 'lyon', 'marseille'))"; SQL statement:

insert into employe values('98765', null, null,null,'DeadLand',null [23513-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Check constraint violation: "(LOWER(VALUE) IN('paris', 'lyon', 'marseille'))"; SQL statement:

insert into employe values('98765', null, null,null,'DeadLand',null [23513-214]
' occurred


### Insérer dans la table Employé deux employés avec le même nom et le même prénom.

In [65]:
query="""
insert into employe values ('87695', 'Adam', 'Funk', null,null,null)
"""
execute(connection,query,show=True)

The error 'Unique index or primary key violation: "public.key_emp_INDEX_9 ON public.employe(nome NULLS LAST, prenome NULLS LAST) VALUES ( /* key:1 */ 'Adam', 'Funk')"; SQL statement:

insert into employe values ('87695', 'Adam', 'Funk', null,null,null [23505-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Unique index or primary key violation: "public.key_emp_INDEX_9 ON public.employe(nome NULLS LAST, prenome NULLS LAST) VALUES ( /* key:1 */ 'Adam', 'Funk')"; SQL statement:

insert into employe values ('87695', 'Adam', 'Funk', null,null,null [23505-214]
' occurred


### Proposer une insertion dans la table Grille_SAL qui ne respecte pas la contrainte sur le salaire qui ne doit pas dépasser 90 000.

In [66]:
query="""

insert into grille_sal values ('23432',900000000)
"""
execute(connection,query,show=True)

The error 'Value too long for column "salaire NUMERIC(7, 2)": "900000000 (32)"; SQL statement:


insert into grille_sal values ('23432',90000000 [22001-214]
DETAIL:  org.h2.jdbc.JdbcSQLDataException: Value too long for column "salaire NUMERIC(7, 2)": "900000000 (32)"; SQL statement:


insert into grille_sal values ('23432',90000000 [22001-214]
' occurred


### Proposer une insertion dans la table Projet qui ne respecte pas la contrainte référentielle vers Employe : insérer un responsable de projet qui n'est pas dans la table Employé

In [67]:
query="""

insert into projet values ('99999',null,null,'Paris',null)
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


### Proposer une insertion dans la table Embauche qui ne respecte pas une des contraintes référentielles : par exemple, associer un employé existant à un projet qui n'existe pas.

In [68]:
query="""

insert into embauche values ('22334' , '90909',null ,'Deve')
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(90909 AS NUMERIC(5)))"; SQL statement:


insert into embauche values ('22334' , '90909',null ,'Deve [23506-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(90909 AS NUMERIC(5)))"; SQL statement:


insert into embauche values ('22334' , '90909',null ,'Deve [23506-214]
' occurred


## Suppressions rejetées

In [87]:
schemafile=open("TME9-creations-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

0 rows
0 rows
0 rows


In [70]:
query="""
select * from Employe e
"""
execute(connection,query,show=True)

10 rows, display only 10 rows
|numss|nome     |prenome|numchef|villee   |datenaiss |
|-----------------------------------------------------
|22334|Adam     |Funk   |22334  |Paris    |1982-12-01|
|45566|Rachid   |Allaoui|22334  |Lyon     |1986-04-13|
|77889|Florent  |Girac  |45566  |Marseille|1990-11-04|
|90011|Mayla    |Aoun   |22334  |Lyon     |1987-03-26|
|22233|Christine|Lara   |45566  |Paris    |1982-08-09|
|34445|Amel     |Orlando|22334  |Lyon     |1976-02-14|
|55666|Mohsen   |Charef |45566  |Paris    |1991-05-28|
|77788|Tim      |Arabi  |22334  |Marseille|1984-06-08|
|89990|Fernando |Lopez  |45566  |Lyon     |1993-10-05|
|11122|Ada      |Tan Lee|22334  |Marseille|1994-03-21|


<cursor object at 0x7ff461181990; closed: -1>

### Proposer une suppression de nuplets de Employe qui respecte les contraintes d'intégrité.

In [71]:
query="""
delete from employe where numss=11122
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


### Proposer une suppression de nuplets de Employe qui ne respecte pas une contrainte référentielle d'une autre table. Précisez laquelle puis essayer de contourner les contraintes jusqu'à pouvoir supprimer l'Employe référencé.

In [72]:
query="""
update employe 
set numchef = null 
where numchef = 22334
"""
execute(connection,query,show=True)

5 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [73]:
query="""
update projet
set respproj = null
where respproj = 22334
"""
execute(connection,query,show=True)

2 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [74]:
query="""
delete from employe where numss=22334
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


### Même question pour Projet.

In [75]:
query="""
delete from projet 
where numproj = 12345
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(12345 AS NUMERIC(5)))"; SQL statement:

delete from projet 
where numproj = 1234 [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(12345 AS NUMERIC(5)))"; SQL statement:

delete from projet 
where numproj = 1234 [23503-214]
' occurred


In [76]:
query="""
alter table embauche 
drop constraint fk_emb_proj
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181e40; closed: -1>

In [77]:
query="""
alter table embauche 
add constraint fk_emb_proj 
foreign key (numproj)
references Projet(numproj)
on delete cascade
"""
# not null !!!! another const conflict 
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461181d50; closed: -1>

### Même question pour Grille_sal.

In [78]:
query="""

delete from grille_sal 
where profil = 'Deve'
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('Deve')"; SQL statement:


delete from grille_sal 
where profil = 'Dev [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('Deve')"; SQL statement:


delete from grille_sal 
where profil = 'Dev [23503-214]
' occurred


In [79]:
query="""
alter table embauche 
drop constraint fk_emb_sal
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461182020; closed: -1>

In [80]:
query="""
alter table embauche 
add constraint fk_emb_sal 
foreign key (profil)
references grille_sal(profil)
on delete cascade
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461182110; closed: -1>

In [81]:
query="""
delete from grille_sal 
where profil = 'Deve'
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


### Est-ce possible de supprimer les nuplets de Embauche sans rencontrer les mêmes problèmes que précédemment ?

<div style=" color:green; font-style:italic; "> Oui, c’est possible car la table **Embauche** n’est référencée par aucune autre table (pas de contrainte de clé étrangère entrante).



In [ ]:
query="""

"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7e92003a7b50; closed: -1>

## Mises à jour rejetées

### Proposez un mise à jour de Employe qui ne respecte pas les contraintes référentielles puis tenter de trouver une mise-a-jour qui les respecte toutes.

In [95]:
query="""
update employe 
set numss = 11111
where numss = 77889
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_emp: public.embauche FOREIGN KEY(numss) REFERENCES public.employe(numss) (CAST(77889 AS NUMERIC(5)))"; SQL statement:

update employe 
set numss = 11111
where numss = 7788 [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_emp: public.embauche FOREIGN KEY(numss) REFERENCES public.employe(numss) (CAST(77889 AS NUMERIC(5)))"; SQL statement:

update employe 
set numss = 11111
where numss = 7788 [23503-214]
' occurred


In [142]:
query="""
alter table embauche 
drop constraint fk_emb_emp
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461134130; closed: -1>

In [144]:
query="""
alter table embauche 
add constraint fk_emb_emp foreign key (numss) references employe(numss) on update cascade on delete cascade
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461137a60; closed: -1>

In [136]:
query="""
update employe 
set numss = 11111
where numss = 77889
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [111]:
query="""
select * from embauche
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|numss|numproj|dateemb   |profil|
|--------------------------------
|11111|12345  |2014-03-01|Deve  |
|90011|12345  |2014-05-01|Tech  |
|22233|75777  |2014-03-01|Deve  |


<cursor object at 0x7ff461182b60; closed: -1>

### Même question avec Projet.

In [114]:
query="""
update projet 
set numproj = 11111
where numproj = 75777
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(75777 AS NUMERIC(5)))"; SQL statement:

update projet 
set numproj = 11111
where numproj = 7577 [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_proj: public.embauche FOREIGN KEY(numproj) REFERENCES public.projet(numproj) (CAST(75777 AS NUMERIC(5)))"; SQL statement:

update projet 
set numproj = 11111
where numproj = 7577 [23503-214]
' occurred


In [115]:
query="""
alter table embauche 
drop constraint fk_emb_proj
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611357b0; closed: -1>

In [116]:
query="""
alter table embauche 
add constraint fk_emb_proj foreign key (numproj) references projet(numproj) on update cascade
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff4611356c0; closed: -1>

In [118]:
query="""
update projet 
set numproj = 11111
where numproj = 75777
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [119]:
query="""
select *
from projet
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|numproj|nomproj|respproj|villep|budget|
|---------------------------------------
|12345  |ADOOP  |22334   |Paris |120000|
|11111  |SKALA  |45566   |Lyon  |180000|
|89097  |BAJA   |22334   |Paris |24000 |


<cursor object at 0x7ff461135d50; closed: -1>

### Proposez une mise-a-jour de Projet qui ne respecte pas la contrainte resprojet selon laquelle le responsable d'un projet habite la ville du projet dont il est responsable.

In [ ]:
query="""

"""
execute(connection,query,show=True)

### Proposez une mise-a-jour de profil dans Grille_sal qui ne respecte pas les contraintes référentielles.

In [122]:
query="""
update grille_sal
set profil = 'nigga'
where profil = 'Tech'
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('Tech')"; SQL statement:

update grille_sal
set profil = 'nigga'
where profil = 'Tech [23503-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "fk_emb_sal: public.embauche FOREIGN KEY(profil) REFERENCES public.grille_sal(profil) ('Tech')"; SQL statement:

update grille_sal
set profil = 'nigga'
where profil = 'Tech [23503-214]
' occurred


In [123]:
query="""
alter table embauche 
drop constraint fk_emb_sal
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461134a90; closed: -1>

In [124]:
query="""
alter table embauche 
add constraint fk_emb_sal foreign key (profil) references grille_sal(profil) on update cascade
"""
execute(connection,query,show=True)

0 rows


<cursor object at 0x7ff461136e30; closed: -1>

In [125]:
query="""
update grille_sal 
set profil = 'Nigga'
where profil = 'Tech'
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [126]:
query="""
select * from embauche
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|numss|numproj|dateemb   |profil|
|--------------------------------
|11111|12345  |2014-03-01|Deve  |
|90011|12345  |2014-05-01|Nigga |
|22233|11111  |2014-03-01|Deve  |


<cursor object at 0x7ff461135f30; closed: -1>

## Exercie TME: Partie 2

Le but de cette partie d'illustrer un cas où les mises-a-jour ne sont pas rejetées mais plutôt propagées.


In [127]:
schemafile=open("TME9-creations-cascade-H2.sql",mode="r",encoding='utf-8')
datafile=open("TME9-insertions-H2.sql",mode="r",encoding='utf-8')
create_schema=schemafile.read()
insert_data=datafile.read()
execute(connection,"drop all objects")
execute(connection, create_schema)
execute(connection, insert_data)
connection.commit()

0 rows
0 rows
0 rows


In [137]:
query="""
select * from Employe
"""
execute(connection,query,show=True)

10 rows, display only 10 rows
|numss|nome     |prenome|numchef|villee   |datenaiss |
|-----------------------------------------------------
|22334|Adam     |Funk   |22334  |Paris    |1982-12-01|
|45566|Rachid   |Allaoui|22334  |Lyon     |1986-04-13|
|11111|Florent  |Girac  |45566  |Marseille|1990-11-04|
|90011|Mayla    |Aoun   |22334  |Lyon     |1987-03-26|
|22233|Christine|Lara   |45566  |Paris    |1982-08-09|
|34445|Amel     |Orlando|22334  |Lyon     |1976-02-14|
|55666|Mohsen   |Charef |45566  |Paris    |1991-05-28|
|77788|Tim      |Arabi  |22334  |Marseille|1984-06-08|
|89990|Fernando |Lopez  |45566  |Lyon     |1993-10-05|
|11122|Ada      |Tan Lee|22334  |Marseille|1994-03-21|


<cursor object at 0x7ff4611363e0; closed: -1>

## Mises-à-jour

### Reéxécutez les mise-à-jour et les suppressions de la Partie 1 et verifiez ce qui se passe.

### Mettez à jour le numéro d'un employé reférencé par d'autres nuplets

In [138]:
query="""
update Employe 
set numss = 99999
where numss = 22233
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [140]:
query="""
select * from embauche
"""
execute(connection,query,show=True)

3 rows, display only 10 rows
|numss|numproj|dateemb   |profil|
|--------------------------------
|11111|12345  |2014-03-01|Deve  |
|90011|12345  |2014-05-01|Tech  |
|99999|75777  |2014-03-01|Deve  |


<cursor object at 0x7ff461137970; closed: -1>

## Suppression

### Supprimer les nuplets de Employe et vérifier que tous les nuplets des autres tables qui référencent des employes venant d'être supprimés seront supprimés eux aussi.

In [145]:
query="""
delete from Employe where numss = 99999
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
The error ''NoneType' object is not iterable' occurred


In [147]:
query="""
select * from embauche
"""
execute(connection,query,show=True)

2 rows, display only 10 rows
|numss|numproj|dateemb   |profil|
|--------------------------------
|11111|12345  |2014-03-01|Deve  |
|90011|12345  |2014-05-01|Tech  |


<cursor object at 0x7ff4611344f0; closed: -1>

# Fermer la connexion

In [148]:
connection.commit() # implicit avec close
connection.close()

In [149]:
connection

<connection object at 0x7ff4615ac900; dsn: 'user=user password=xxx dbname=jo8082 host=localhost port=8082', closed: 1>